In [7]:
import numpy as np
import jax
import jax.numpy as jnp


In [15]:
import jax
import jax.numpy as jnp
import haiku as hk
import optax

import pandas as pd
import pickle

from jax import config

config.update("jax_default_matmul_precision", "float32")



class CNNAffinityPredictor(hk.Module):
    def __init__(self, seq_hidden=128, prop_hidden=32, fc_hidden=[64, 32], name=None):
        super().__init__(name=name)
        self.seq_hidden = seq_hidden
        self.prop_hidden = prop_hidden
        self.fc_hidden = fc_hidden

    def __call__(self, seq_onehot ):
        """
        seq_onehot: [batch_size, seq_len, num_amino_acids]
        """
        # ---- Sequence branch: CNN ----
        x = seq_onehot  # [B, L, A]

        # First 1D convolution (along sequence length)
        conv1 = hk.Conv1D(output_channels=64, kernel_shape=3, stride=1, padding="SAME")
        x = conv1(x)
        x = jax.nn.relu(x)

        # Second 1D convolution
        conv2 = hk.Conv1D(
            output_channels=self.seq_hidden, kernel_shape=3, stride=1, padding="SAME"
        )
        x = conv2(x)
        x = jax.nn.relu(x)

        # Third 1D convolution
        conv3 = hk.Conv1D(
            output_channels=self.seq_hidden, kernel_shape=3, stride=1, padding="SAME"
        )
        x = conv3(x)
        x = jax.nn.relu(x)

        # Global mean pooling along sequence length
        seq_vector = jnp.mean(x, axis=1)  # [B, seq_hidden]
        
        # ---- Fully connected layers ----
        fc = seq_vector
        for hidden in self.fc_hidden:
            fc = hk.Linear(hidden)(fc)
            fc = jax.nn.relu(fc)

        # ---- Output layer ----
        affinity = hk.Linear(1)(fc)  # scalar output
        return affinity


# Haiku transform
def model_fn(seq_onehot ):
    model = CNNAffinityPredictor()
    return model(seq_onehot )


# ---- actual data ----

# TO DO - this is actually also alr. in colabfold ...
# from colabdesign.af.alphafold.common.residue_constants import * 
def pad_sequences(sequences, max_len=15, alphabet="ACDEFGHIKLMNPQRSTVWY"):
    """
    sequences: list of sequences of different lengths, each as a string (e.g. 'ACDE')
    max_len: length to pad/truncate sequences to
    alphabet: string of possible amino acids (default 20 canonical)

    returns: [batch, max_len, num_amino_acids] one-hot encoded
    """

    # map amino acid letters to indices
    aa_to_idx = {aa: i for i, aa in enumerate(alphabet)}
    num_amino_acids = len(alphabet)

    batch_size = len(sequences)
    padded = jnp.zeros((batch_size, max_len), dtype=jnp.int32)

    for i, seq in enumerate(sequences):
        indices = seq
        if type(seq[0]) is not int:  # if input is already indices
            indices = [aa_to_idx.get(aa, 0) for aa in seq]  # default to 0 if unknown
        # print(')

        padded = padded.at[i, : len(indices)].set(jnp.array(indices, dtype=jnp.int32))
    # one-hot encode
    onehot = jax.nn.one_hot(padded, num_classes=num_amino_acids, dtype=jnp.float32)
    return onehot


# ---- Loss and training step ----
def loss_fn(params, seq_onehot , targets):
    preds = model.apply(params, rng, seq_onehot )
    return jnp.mean((preds - targets) ** 2)


@jax.jit
def train_step(params, opt_state, seq_onehot , targets):
    grads = jax.grad(loss_fn)(params, seq_onehot , targets)
    updates, opt_state = optimizer.update(grads, opt_state)
    params = optax.apply_updates(params, updates)
    loss = loss_fn(params, seq_onehot, targets)
    return params, opt_state, loss




def init_surr_model():
    """
    reads MD_data
    initializes model

    Returns:
        model (class?): initialized model
        params_restored (dic?): trained parameter of model
        rng (): key
    """
    model = hk.transform(model_fn)

    # ---- data ----
    df = pd.read_csv("/home/kunzj/bindcraft_modified/flexs/landscape.csv")


    sequences = df["sequence"].tolist()
    seq_onehot = pad_sequences(sequences=sequences)

    rng = jax.random.PRNGKey(42)

    model.init(rng, seq_onehot )

    print("start loading miodel weights")
    with open("/home/kunzj/BindCraft_uva_internship/surr_model_params/params_only_seq.pkl", "rb") as f:
        params_restored = pickle.load(f)

    return model, params_restored, rng


In [16]:
import numpy as np

arr = np.array([
    [[-2.36080005e-03, -3.72781442e-03,  7.69840740e-03,  9.97477118e-03,
      -1.39921037e-02,  6.40460243e-03, -1.54027296e-03,  1.09545197e-02,
       8.68443213e-03,  2.49815136e-02, -2.92576849e-03, -6.35457272e-03,
      -1.80691686e-02, -7.44924415e-03, -1.41751431e-02,  1.31367743e-02,
       2.19815644e-03, -2.12006806e-03, -5.27540036e-03, -1.54052228e-02],
     [ 8.79693497e-03,  1.65762613e-04, -9.73921327e-04, -1.12769485e-03,
       1.82041638e-02,  5.45475399e-03, -6.26335945e-03, -4.40226495e-03,
       8.89710709e-03,  1.97528279e-04, -4.79179295e-03,  2.68441252e-03,
       3.26145999e-03, -1.68932974e-03, -1.91374999e-02,  3.47616756e-03,
       2.16571405e-03, -1.01681529e-02, -2.75591668e-03,  3.33108776e-03],
     [-6.18600659e-03,  1.31683087e-03,  6.07772963e-03, -1.10447044e-02,
      -2.32055485e-02,  5.85484039e-03, -9.95235238e-03,  2.06615124e-03,
       1.48891378e-02, -1.12352986e-03,  1.30031407e-02,  2.37367535e-03,
      -1.72690433e-02, -4.37916396e-03, -9.60247219e-03, -6.08911412e-03,
       3.10531911e-03,  1.65856723e-02, -5.37615176e-03, -2.21851859e-02],
     [-3.85603262e-03, -1.60016287e-02, -1.69904809e-02, -5.10704238e-03,
      -1.97748691e-02,  1.29161519e-03, -2.84488453e-03, -3.00132274e-03,
       1.33437887e-02, -4.14031651e-03,  9.17649921e-03,  1.43130366e-02,
       2.87926174e-03, -1.08938077e-02, -2.59338669e-03, -9.08615999e-03,
      -1.29437195e-02,  2.12136516e-03, -1.58831608e-02, -1.29286144e-02],
     [ 1.23669086e-02, -1.28961792e-02,  9.50582232e-03, -3.12453415e-02,
      -1.49833225e-02,  1.22713251e-02, -2.24210322e-04, -3.01675871e-03,
      -5.04165562e-03,  2.16187886e-03, -9.93701909e-03, -7.84899457e-04,
       7.83239957e-03,  1.98824387e-02,  1.79558415e-02, -2.01171599e-02,
       1.47443323e-03,  9.89587326e-03, -2.45432113e-03,  1.76025946e-02],
     [ 1.12711564e-02, -1.33253937e-03,  8.43048003e-03, -1.61641408e-02,
       1.94719415e-02,  1.55795030e-02,  5.51314931e-03, -1.37260305e-02,
      -2.18912065e-02, -1.65151563e-02, -2.87744077e-03,  8.82301666e-03,
      -1.34852808e-03, -2.13954095e-02,  6.66049495e-03,  3.46347224e-03,
       8.14057700e-03, -1.83644723e-02,  3.06505268e-03, -9.47629567e-03],
     [ 8.50125775e-03,  8.66916962e-03,  3.36668221e-03, -5.99051127e-03,
      -1.21711819e-02, -3.58916679e-03, -1.57902436e-03,  2.72700656e-03,
      -6.94693113e-03,  4.83650481e-03, -3.41619318e-03,  1.97472004e-03,
       1.46966241e-03, -1.34587931e-02,  1.60506852e-02, -4.71630180e-03,
      -2.92341737e-03, -7.24733714e-03,  2.56623444e-03,  3.86810815e-03],
     [ 3.39446333e-03, -1.75965414e-03, -1.89893588e-04,  2.46516690e-02,
      -4.13947506e-03,  1.38971247e-02,  1.94974616e-02, -6.57330733e-03,
      -4.86981031e-03, -2.65155197e-03,  1.30805606e-03, -1.79737024e-02,
       5.05894143e-03, -2.87281233e-04,  1.68374286e-03, -4.29870002e-03,
      -8.99066217e-03, -5.13282744e-03, -9.05575790e-03, -1.33222360e-02],
     [-9.74831358e-03,  1.33369500e-02, -6.36692019e-03,  1.16440523e-02,
      -7.07692874e-04, -1.58758443e-02,  2.79189553e-02, -4.66680620e-03,
       2.89249321e-04, -5.38470550e-03,  1.20872874e-02, -3.60692525e-03,
       2.53618918e-02,  5.64440573e-03, -1.62934442e-03, -1.87306423e-02,
       3.21786013e-03, -8.66451487e-03,  5.80744236e-05, -3.38110025e-03],
     [ 1.43663473e-02,  7.80178932e-03,  8.97583365e-03,  9.39636026e-03,
       1.41970729e-02,  3.22585343e-03,  3.63368657e-03, -1.49487238e-02,
       9.16504068e-04,  1.28504597e-02, -1.67835306e-03, -9.25866608e-03,
       2.29982147e-03, -4.44918592e-03,  1.14191289e-03,  6.24152971e-03,
      -1.14267098e-03,  2.48369128e-02,  1.64313335e-02,  9.20163747e-03],
     [-7.64335925e-03,  2.56234920e-03,  3.35054612e-03,  4.62377118e-03,
       1.20515246e-02,  1.72210000e-02,  3.39777092e-03,  1.16544664e-02,
       2.86216382e-02, -4.94851312e-03, -1.14778038e-02, -7.87213072e-03,
      -1.71467029e-02,  1.27071980e-02, -1.55193945e-02, -4.51314496e-03,
       7.11307954e-03,  2.81712878e-03, -1.01058679e-02,  8.43480974e-03],
     [-2.97919079e-03, -7.00903963e-03, -2.75061410e-02, -6.98372209e-03,
      -5.80575783e-04,  3.99786383e-02, -1.74830034e-02, -1.24292006e-03,
       5.02731418e-03, -1.14588942e-02,  2.36864965e-02, -4.66925092e-03,
       7.27940258e-03, -6.72494667e-03, -4.71683685e-03,  1.14128944e-02,
      -8.11732188e-03,  1.45297367e-02,  4.09388077e-03,  1.14323548e-03],
     [ 1.72032248e-02,  2.12313171e-04, -8.22037552e-03, -7.37313880e-03,
       5.84334880e-03, -4.83200094e-03,  1.67690199e-02, -7.87221175e-03,
       6.80305576e-03, -2.57609237e-04, -1.15316184e-02,  1.22672822e-02,
      -1.13119408e-02, -1.04966313e-02,  1.18517531e-02,  1.65678989e-02,
       3.29727703e-03,  2.22567078e-02, -1.24000879e-02,  6.27109827e-03],
     [-1.51805999e-02,  1.01516368e-02,  8.59795511e-03, -9.16468818e-03,
      -4.57124843e-04,  2.01089457e-02, -8.07815976e-03,  9.51592904e-03,
       8.84950720e-03,  3.18939984e-03,  9.03282594e-03,  5.26199723e-03,
       9.47503746e-03,  8.06806982e-03, -1.21590495e-02, -1.15282172e-02,
      -1.40316142e-02,  2.63778173e-04, -4.63074958e-03, -1.26996601e-04],
     [-8.97699408e-03,  4.58509196e-03, -5.77258586e-04, -1.46099832e-02,
      -1.93490088e-02,  9.42331459e-03,  1.79096926e-02, -1.47572802e-02,
      -1.04613164e-02, -2.49794452e-03,  4.30959696e-03, -1.54773351e-02,
       1.47204206e-04,  8.12424161e-03, -1.02441963e-02,  4.45047440e-03,
       4.45192214e-03, -8.72448145e-04,  2.36343220e-02,  7.62885669e-03]]
])

print(arr.shape)


(1, 15, 20)


In [17]:
model, params_restored, rng = init_surr_model()

start loading miodel weights


In [20]:
aff_val =model.apply(params_restored,rng,arr)

In [ ]:
float(aff_val[0,0])

TypeError: Only scalar arrays can be converted to Python scalars; got arr.ndim=2